### Imports

In [5]:
# Imports
import gurobipy as gp
from gurobipy import GRB
import json

### Optimization function

In [ ]:
import gurobipy as gp
from gurobipy import GRB

def maximize_excess_funds(T, n, c, r, beta, r1, r2, r3):
    model = gp.Model("cash_flow")

    # Sets (time is 1..T for cleaner constraints)
    months = range(1, T + 1)
    lines = range(n)

    # Decision variables
    borrow = model.addVars(lines, months, lb=0.0)
    cash = model.addVars(months, lb=0.0)         

    # Last usable month for each line so all repayments finish by T
    last_use = []
    for i in lines:
        max_lag = 0
        if r1[i] > 0: max_lag = max(max_lag, 1)
        if r2[i] > 0: max_lag = max(max_lag, 2)
        if r3[i] > 0: max_lag = max(max_lag, 3)
        last_use.append(T - max_lag if max_lag > 0 else T)

    # Per-month borrowing caps
    for i in lines:
        for t in months:
            borrow[i, t].UB = beta[i] if t <= last_use[i] else 0

    # Cash flow 
    for t in months:
        curr_cash = gp.quicksum(borrow[i, t] for i in lines) 
        prev_cash   = (1.0 + r) * (cash[t-1] if t-1 >= 1 else 0)
        R_1  = gp.quicksum(r1[i] * borrow[i, t-1] for i in lines) if t-1 >= 1 else 0.0
        R_2  = gp.quicksum(r2[i] * borrow[i, t-2] for i in lines) if t-2 >= 1 else 0.0
        R_3  = gp.quicksum(r3[i] * borrow[i, t-3] for i in lines) if t-3 >= 1 else 0.0
    
        model.addConstr(
            cash[t] == prev_cash + float(c[t-1]) + curr_cash - (R_1 + R_2 + R_3),
            name=f"cash_flow[{t}]"
        )

    # Objective: maximize ending cash s[T]
    model.setObjective(cash[T], GRB.MAXIMIZE)

    model.optimize()
    print(model.Status)
    return model.ObjVal

### Inputs

Input using input bar:

In [ ]:
T = input("Input T: int")
n = input("Input n: int")
c = input("Input c: arr")
r = input('Input r: float')
beta = input('Input beta: arr')
r1 = input('Input r1: arr')
r2 = input('Input r2: arr')
r3 = input('Input r3: arr')

Input using data:

In [30]:
data = {
    "T": 5,
    "n": 2,
    "c":  [10, 5, 0, 0, 0],  
    "r":  0.01,                            
    "beta":[5, 4],                      
    "r1": [1.1, 1.05],
    "r2": [0.1, 0.2],
    "r3": [0.0, 0.0],
}
T = int(data["T"])
n = int(data["n"])
c = list(map(float, data["c"]))
r = float(data["r"])
beta = list(map(float, data["beta"]))
r1 = list(map(float, data["r1"]))
r2 = list(map(float, data["r2"]))
r3 = list(map(float, data["r3"]))

### Run code

In [31]:
result = maximize_excess_funds(T, n, c, r, beta, r1, r2, r3)
print(f"Maximum excess funds: {result}")

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 5 rows, 15 columns and 33 nonzeros
Model fingerprint: 0xf92f06b1
Coefficient statistics:
  Matrix range     [1e-01, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [4e+00, 5e+00]
  RHS range        [5e+00, 1e+01]
Presolve removed 3 rows and 8 columns
Presolve time: 0.00s
Presolved: 2 rows, 7 columns, 12 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.4764184e+01   2.594626e+00   0.000000e+00      0s
       2    1.5557545e+01   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.555754510e+01
2
Maximum excess funds: 15.557545099999999
